In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import sys, os

# Adjust these paths to match your project
sys.path.append('../src/')  # <-- change this

from dataloaders import PREPROC_MAPPER
from dataloaders.utils import get_dataset, ZeroShotSamplerReduced
from dataloaders import DiskDatasetDiv
from torch.utils.data import Subset

plt.style.use('dark_background')

In [6]:
# --- Config: adjust these to match your yaml/setup ---
DATA_BASE       = '/path/to/data/'       # cb.data_base
DATASET_NAME    = 'your_dataset_name'    # item["name"]
TEMPORAL_BUNDLE = 4                      # cm.temporal_bundling
TRAIN_RATIO     = 0.8                    # ct.train_ratio
SEED            = 42                     # cd.seed
TIMESAMPLE      = 1                      # item["timesample"]


preproc_path = DATA_BASE + 'preproc_' + DATASET_NAME
dataset = DiskDatasetDiv(preproc_path, temporal_bundling=TEMPORAL_BUNDLE, forward_steps=1)

val_sampler = ZeroShotSamplerReduced(
    dataset, train_ratio=TRAIN_RATIO,
    split="val", seed=SEED, skip_timesteps=TIMESAMPLE
)
val_subset = Subset(dataset, val_sampler.indices)

print(f"Dataset loaded: {len(val_subset)} validation samples")
print(f"Data shape: {dataset.datashape}")
print(f"Mean: {dataset.avg:.4f}, Std: {dataset.std:.4f}")

FileNotFoundError: Metadata file /path/to/data/preproc_your_dataset_name/meta.h5 does not exist. Please preprocess the data first.

In [7]:
# Pick a random trajectory index from validation set
traj_idx = val_sampler.random_val_traj()
traj = dataset.get_single_traj(traj_idx)  # shape: (T, C, H, W)

global_mean = dataset.avg
global_std  = dataset.std

# Denormalize
traj_denorm = traj.float() * global_std + global_mean

print(f"Trajectory shape: {traj.shape}  (T, C, H, W)")
print(f"Value range after denorm: [{traj_denorm.min():.3f}, {traj_denorm.max():.3f}]")

NameError: name 'val_sampler' is not defined

In [8]:
N_STEPS_TO_PLOT = 6   # how many timesteps to show
CHANNEL_X = 0         # vx channel
CHANNEL_Y = 1         # vy channel

T = traj_denorm.shape[0]
step = max(1, T // N_STEPS_TO_PLOT)
timesteps = list(range(0, T, step))[:N_STEPS_TO_PLOT]

fig, axes = plt.subplots(2, len(timesteps), figsize=(3 * len(timesteps), 6))
fig.suptitle(f"Trajectory: {DATASET_NAME}  |  rows: vx / vy", fontsize=14)

for col, t in enumerate(timesteps):
    frame = traj_denorm[t]  # (C, H, W)

    for row, (ch, label) in enumerate([(CHANNEL_X, 'vx'), (CHANNEL_Y, 'vy')]):
        ax = axes[row, col]
        im = ax.imshow(frame[ch].numpy(), cmap='viridis', origin='lower')
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f"t={t}", fontsize=11)
        if col == 0:
            ax.set_ylabel(label, fontsize=12, rotation=0, labelpad=20)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

NameError: name 'traj_denorm' is not defined

In [9]:
fig, axes = plt.subplots(1, len(timesteps), figsize=(3 * len(timesteps), 3))
fig.suptitle(f"Velocity Magnitude  |  {DATASET_NAME}", fontsize=14)

for col, t in enumerate(timesteps):
    frame = traj_denorm[t]
    mag = torch.sqrt(frame[CHANNEL_X]**2 + frame[CHANNEL_Y]**2).numpy()
    ax = axes[col]
    im = ax.imshow(mag, cmap='inferno', origin='lower')
    ax.set_title(f"t={t}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

NameError: name 'timesteps' is not defined